# 13 — Clean ISIC 2019 master dataset (v2)

This notebook is the only supported replacement for the old HAM10000+ISIC concatenation.

Policy: ISIC 2019 is the master aggregate corpus. HAM10000 is used only for provenance and overlap auditing because its images are already represented in ISIC 2019. The notebook reads the official ISIC 2019 metadata and uses its common lesion identifier for the split.

The notebook never deletes legacy files. It writes a separate data_clean_v2 directory.


In [ ]:
# --- Colab setup ---
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/zkoymen/melanoma-detection-ham10000.git"
REPO_REF = "codex/clean-data-protocol"
PROJECT = Path("/content/melanoma-detection-ham10000")

if not PROJECT.exists():
    subprocess.run(["git", "clone", "--branch", REPO_REF, REPO_URL, str(PROJECT)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT), "fetch", "origin", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(PROJECT), "checkout", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(PROJECT), "pull", "--ff-only", "origin", REPO_REF], check=True)

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = Path("/content/drive/MyDrive/melanoma")
LEGACY_DIR = DRIVE_ROOT / "data"
CLEAN_DIR = DRIVE_ROOT / "data_clean_v2"
ISIC_DIR = Path("/content/isic2019")
ISIC_DIR.mkdir(parents=True, exist_ok=True)

# All later model notebooks will use this explicit versioned directory.
os.environ["MELANOMA_DATA_DIR"] = str(CLEAN_DIR)
os.environ["MELANOMA_DATA_VERSION"] = "clean_v2_isic2019_master"
os.environ["MELANOMA_RESULTS_DIR"] = str(DRIVE_ROOT / "results_clean_v2")
os.environ["MELANOMA_CHECKPOINT_DIR"] = str(DRIVE_ROOT / "checkpoints_clean_v2")
os.environ["MELANOMA_PAPER_DIR"] = str(DRIVE_ROOT / "paper_clean_v2")
os.environ["MELANOMA_LOCAL_CACHE"] = "/content/local_data_clean_v2"
print("Project:", PROJECT)
print("Legacy data (read-only):", LEGACY_DIR)
print("Clean output:", CLEAN_DIR)



In [ ]:
# --- Obtain the official ISIC 2019 metadata ---
# Official source linked from https://challenge.isic-archive.com/data/
import urllib.request
import pandas as pd

METADATA_URL = "https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_Metadata.csv"
METADATA_PATH = ISIC_DIR / "ISIC_2019_Training_Metadata.csv"

if not METADATA_PATH.exists():
    print("Downloading official metadata...")
    urllib.request.urlretrieve(METADATA_URL, METADATA_PATH)

meta = pd.read_csv(METADATA_PATH)
print("Metadata shape:", meta.shape)
print("Columns:", meta.columns.tolist())
assert len(meta) == 25331, "Unexpected metadata row count; stop before training."
assert any(str(c).lower() == "image" for c in meta.columns)
assert any("lesion" in str(c).lower() for c in meta.columns)
print("Official metadata check passed.")



In [ ]:
# --- Read-only input checks ---
required = [
    LEGACY_DIR / "X_all.npy", LEGACY_DIR / "y_all.npy",
    LEGACY_DIR / "ids_all.npy", LEGACY_DIR / "lesion_ids_all.npy",
    LEGACY_DIR / "X_isic2019_mel.npy", LEGACY_DIR / "ids_isic2019_mel.npy",
    LEGACY_DIR / "X_isic2019_nonmel.npy", LEGACY_DIR / "ids_isic2019_nonmel.npy",
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Missing inputs. Run the preprocessing part of the old ISIC notebook once "
        "to create the two ISIC array pairs, then return here. Nothing was deleted.\n"
        + "\n".join(missing)
    )

import numpy as np
print("HAM X:", np.load(LEGACY_DIR / "X_all.npy", mmap_mode="r").shape)
print("ISIC MEL:", np.load(LEGACY_DIR / "X_isic2019_mel.npy", mmap_mode="r").shape)
print("ISIC NON-MEL:", np.load(LEGACY_DIR / "X_isic2019_nonmel.npy", mmap_mode="r").shape)
print("Input check passed.")


In [ ]:
# --- Audit HAM/ISIC overlap before building anything ---
import json
from src.clean_dataset import audit_raw_overlap
AUDIT_PATH = CLEAN_DIR / "raw_overlap_audit.json"
report = audit_raw_overlap(LEGACY_DIR, LEGACY_DIR, METADATA_PATH, AUDIT_PATH)
print(json.dumps({
    k: report[k] for k in [
        "legacy_ham_count", "isic_preprocessed_count",
        "isic_metadata_rows", "exact_image_id_overlap_count",
        "stored_array_hash_overlap_count",
        "isic_ids_missing_from_metadata_rows_count"
    ]
}, indent=2))
assert report["isic_metadata_rows"] == 25331
assert report["isic_ids_missing_from_metadata_rows_count"] == 0
print("ISIC rows with blank lesion_id (image fallback will be used):", report["isic_ids_with_missing_lesion_id_count"])
print("Audit passed: metadata coverage is complete. Review raw_overlap_audit.json before training.")



### Stop-and-review gate

At this point inspect raw_overlap_audit.json. The counts are provenance evidence. Do not estimate leakage from probability formulas; the saved IDs and hashes are the evidence.


In [ ]:
# --- Build clean v2 in its separate directory ---
from src.clean_dataset import build_clean_isic_master_dataset

X, y, ids, idx_train, idx_val, idx_test, manifest = build_clean_isic_master_dataset(
    legacy_dir=LEGACY_DIR,
    isic_dir=LEGACY_DIR,
    metadata_csv=METADATA_PATH,
    output_dir=CLEAN_DIR,
    seed=42,
    train_frac=0.70,
    val_frac=0.15,
)
print(json.dumps(manifest["counts"], indent=2))
print(json.dumps(manifest["checks"], indent=2))
assert manifest["checks"]["missing_metadata_rows"] == 0
assert manifest["checks"]["image_id_unique_within_and_across_splits"]
assert manifest["checks"]["lesion_id_disjoint_across_splits"]
print("CLEAN DATASET READY:", CLEAN_DIR)



In [ ]:
# --- Final integrity check used before any model notebook ---
from src.data import load_arrays_balanced
X, y, ids, tr, va, te = load_arrays_balanced(CLEAN_DIR, use_local_cache=False)
lesions = np.load(CLEAN_DIR / "lesion_combined.npy", allow_pickle=True)

assert len(set(ids.tolist())) == len(ids)
assert set(lesions[tr].tolist()).isdisjoint(set(lesions[va].tolist()))
assert set(lesions[tr].tolist()).isdisjoint(set(lesions[te].tolist()))
assert set(lesions[va].tolist()).isdisjoint(set(lesions[te].tolist()))
assert set(ids[tr].tolist()).isdisjoint(set(ids[va].tolist()))
assert set(ids[tr].tolist()).isdisjoint(set(ids[te].tolist()))
assert set(ids[va].tolist()).isdisjoint(set(ids[te].tolist()))

print("X:", X.shape)
print("Train/Val/Test:", len(tr), len(va), len(te))
for name, idx in [("train", tr), ("validation", va), ("test", te)]:
    print(name, "mel/non-mel:", int((y[idx] == 1).sum()), int((y[idx] == 0).sum()),
          "unique lesions:", len(set(lesions[idx].tolist())))
print("All clean split assertions passed.")



## Next notebooks

Run notebooks 01–11 only after this notebook finishes successfully.

Before opening them, set:

    import os
    os.environ["MELANOMA_DATA_DIR"] = "/content/drive/MyDrive/melanoma/data_clean_v2"

The existing model notebooks will then read the clean arrays from data_clean_v2. Never point them back to the legacy data directory for the corrected results.
